# Data Preparation

## Objective

Clean train, features, and stores based on the issues found in 01_data_understanding.ipynb (MarkDown missingness, negative/zero Weekly_Sales, string dates), then merge them into a single store-week table ready for EDA and feature engineering. No cleaning decision here is arbitrary — each one is tied back to what was actually observed in validation. 

In [3]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv", parse_dates=["Date"])
features = pd.read_csv("../data/raw/features.csv", parse_dates=["Date"])
stores = pd.read_csv("../data/raw/stores.csv")

print(train.shape, features.shape, stores.shape)

(421570, 5) (8190, 12) (45, 3)


## Handling MarkDown Missingness

**Observed:** MarkDown1-5 are missing in 50.8% to 64.3% of features rows (from 01_data_understanding.ipynb). Markdown promotions in this dataset are known to have started partway through the data collection window, not from day one.

**Decision:** Treat missing MarkDown values as "no active promotional markdown that week," not as a value to impute from surrounding data. Fill NaN with 0.

**Why:** Imputing a non-zero value (mean/median/interpolation) would fabricate promotional activity that likely didn't happen. Zero is the only value consistent with "no markdown recorded." This also keeps the feature meaningful for later modelling: 0 means no markdown, any positive number means an active markdown amount.

In [5]:
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
features[markdown_cols] = features[markdown_cols].fillna(0)

print(features[markdown_cols].isnull().sum())

MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64


## Handling CPI / Unemployment Missingness

**Observed:** 585 rows (~7% of features) are missing CPI and Unemployment, concentrated in the later weeks (features extends to 2013-07-26, beyond train's 2012-10-26 end date).

**Decision:** Forward-fill CPI and Unemployment per store, ordered by date.

**Why:** These are slow-moving macroeconomic indicators (monthly/quarterly in reality, reported weekly here) — the most recent known value is a reasonable stand-in for a few missing weeks, unlike MarkDown which can genuinely be zero. This is not target leakage since CPI/Unemployment are external economic indicators, not derived from Weekly_Sales.

In [7]:
features = features.sort_values(["Store", "Date"])
features[["CPI", "Unemployment"]] = features.groupby("Store")[["CPI", "Unemployment"]].ffill()

print(features[["CPI", "Unemployment"]].isnull().sum())

CPI             0
Unemployment    0
dtype: int64


## Handling Negative and Zero Weekly_Sales

In [9]:
negative_sales = train[train["Weekly_Sales"] < 0]

print("Number of negative rows:", len(negative_sales))
print("As % of total rows:", round(len(negative_sales) / len(train) * 100, 2))
print()
print("Distribution of negative values:")
print(negative_sales["Weekly_Sales"].describe())
print()
print("Number of distinct Store-Dept combos affected:", negative_sales[["Store","Dept"]].drop_duplicates().shape[0])
print("Number of distinct Depts affected:", negative_sales["Dept"].nunique())

Number of negative rows: 1285
As % of total rows: 0.3

Distribution of negative values:
count    1285.000000
mean      -68.608218
std       231.664245
min     -4988.940000
25%       -41.000000
50%       -13.200000
75%        -4.940000
max        -0.020000
Name: Weekly_Sales, dtype: float64

Number of distinct Store-Dept combos affected: 376
Number of distinct Depts affected: 50


**Observed:** 1,285 rows (0.3% of train) have negative Weekly_Sales, spread across 376 distinct Store-Dept combinations and 50 distinct departments. The distribution is mostly small values (median -13.20, 75th percentile -4.94) with a long tail down to a minimum of -4,988.94 — a handful of large return weeks rather than a uniform pattern across the data.

**Decision:** Keep negative Weekly_Sales values as-is; do not delete or clip them.

**Why:** Weekly_Sales is department-level revenue net of returns for that week. A negative value is a real business event (returns exceeding new sales), not a data entry error — deleting these rows would understate return-heavy weeks and bias any store/department aggregate. Zero-sales weeks (73 rows) are also kept, since a department can legitimately sell nothing in a slow week. Since these are spread across only 376 of the ~4,000+ store-dept combinations and are small in magnitude for most cases, they will naturally get smoothed out once we aggregate to store-level weekly totals in the next step.

## Merging into a Single Table

In [12]:
merged = train.merge(features, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
merged = merged.merge(stores, on="Store", how="left")

print(merged.shape)
print(merged.columns.tolist())

(421570, 17)
['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday_feat', 'Type', 'Size']


In [14]:
print("IsHoliday mismatches:", (merged["IsHoliday"] != merged["IsHoliday_feat"]).sum())

IsHoliday mismatches: 0


In [16]:
merged = merged.drop(columns=["IsHoliday_feat"])
print(merged.isnull().sum().sum())

0


In [18]:
merged.to_csv("../data/processed/merged_store_dept_week.csv", index=False)
print("Saved:", merged.shape)

Saved: (421570, 16)


## Summary

Cleaned and merged train, features, and stores into a single store-dept-week table (421,570 rows, 17 columns). MarkDown1-5 NaNs filled with 0 (no active promotion). CPI/Unemployment NaNs forward-filled per store. Negative and zero Weekly_Sales retained as genuine business events. Merge verified: no row count change, no IsHoliday mismatch between sources, no new nulls introduced.

Saved to data/processed/merged_store_dept_week.csv — this is the input for the next notebook (03_time_series_analysis.ipynb), where it will be aggregated to store-level weekly totals for trend/seasonality investigation.